# 12.5 - Docker & Containerization

**Module 12 - Production Deployment** | Netsetos GenAI Engineering

This notebook builds the whole craft of shipping an AI app in a container - without a Docker daemon, without an API key, using tiny Python models of each piece. You'll simulate environment drift, the layer-and-cache build model, base-image tradeoffs, multi-stage builds, reproducible+fast dependencies with uv, non-root/PID-1/healthcheck safety, and the build->scan->push->run pipeline.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the two third-party packages the notebook's examples reference. Every cell here is pure Python simulation, so this is a light dependency set - leave it commented locally and uncomment on Colab or a fresh environment.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install numpy python-dotenv -q  # noqa


**Explanation**

A one-line environment-prep cell, commented out by default so it does not run on a machine that already has these packages.

**How the code works - step by step**
- **`numpy`** - numerical library; the demos only reference it by name/version, they don't compute with it.
- **`python-dotenv`** - lets you load API keys from a `.env` file in the next cell.
- The `-q` flag keeps pip quiet; `# noqa` marks the `!pip` magic so linters ignore it.

**In one line:** optional package install for a fresh environment.

**What you'll see:** (no output - environment prep)

## Environment keys (optional here)

Load any provider API keys from the environment. None of the cells in this lesson actually call an API - it's all keyless simulation - but this mirrors the standard notebook opening so the pattern is familiar.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

A configuration cell that seeds three provider-key slots from the environment without overwriting anything already set. It's a placeholder for the standard secure-key pattern, not something these simulations need.

**How the code works - step by step**
- **`import os`** - gives access to the process environment.
- **`os.environ.setdefault("OPENAI_API_KEY", "")`** - sets an empty default only if the key is not already present (setdefault never clobbers a real value).
- Same for **`ANTHROPIC_API_KEY`** and **`GOOGLE_API_KEY`**, each annotated with where to get one.

**In one line:** never hardcode keys - read them from the environment; not required for this lesson's cells.

**What you'll see:** (no output - environment prep)

## 1 - Kill "works on my machine": freeze the environment

The problem the whole lesson solves. Your code depends on a specific Python version and specific library versions; when the prod box has drifted, the app crashes in a way the laptop never reproduces. This cell simulates that failure, then shows how freezing one environment into an image makes both hosts pass.

In [ ]:
# Same code, different machines. Only a FROZEN image runs identically everywhere.
def build_env(python, os_name, numpy):
    return {"python": python, "os": os_name, "numpy": numpy}

def run_app(env):
    # The app was written against numpy 2.2 (it calls a 2.2-only API).
    py_minor = int(env["python"].split(".")[1])
    nx = tuple(int(p) for p in env["numpy"].split(".")[:2])
    if py_minor < 12:
        return "CRASH: needs Python 3.12+, found " + env["python"]
    if nx < (2, 2):
        return "CRASH: app uses a numpy 2.2 API, found " + env["numpy"]
    return "OK: served a request on " + env["os"]

dev  = build_env("3.13.1", "macOS-arm64",   "2.2.2")
prod = build_env("3.12.3", "linux-x86_64",  "2.0.1")   # a different box, drifted deps

print("Without a container - the SAME code on two machines:")
print("  dev  ", run_app(dev))
print("  prod ", run_app(prod))

# Containerize: freeze python + os + deps into ONE image, then run that image anywhere.
IMAGE = build_env("3.13.1", "linux (image)", "2.2.2")  # the image IS the environment
print("With a container - the SAME image on two machines:")
print("  dev-host  ", run_app(IMAGE))
print("  prod-host ", run_app(IMAGE))
print()
print("An image freezes filesystem + runtime + deps, so it runs the same everywhere.")

# Output:
# Without a container - the SAME code on two machines:
#   dev   OK: served a request on macOS-arm64
#   prod  CRASH: app uses a numpy 2.2 API, found 2.0.1
# With a container - the SAME image on two machines:
#   dev-host   OK: served a request on linux (image)
#   prod-host  OK: served a request on linux (image)
#
# An image freezes filesystem + runtime + deps, so it runs the same everywhere.

**Explanation**

A tiny model of a deploy, not a real one: `run_app` is a compatibility checker that returns CRASH or OK given an environment dict. The point is the contrast between running the same *code* on two drifted machines versus running the same *image* on both.

**How the code works - step by step**
- **`build_env`** - packs a python version, OS name, and numpy version into an environment dict (a stand-in for a machine's real environment).
- **`run_app`** - the app's requirements encoded as checks: it needs Python 3.12+ and a numpy 2.2 API, and returns a CRASH string if either is too old, else OK.
- **`dev` vs `prod`** - two environments that have drifted apart (prod is older Python and older numpy).
- **`IMAGE`** - one frozen environment; running that same image on both hosts passes on both.

**In one line:** without a container the same code diverges across machines; a frozen image runs identically everywhere.

**What you'll see:** Without a container, dev prints OK but prod prints "CRASH: app uses a numpy 2.2 API, found 2.0.1"; with a container, both dev-host and prod-host print OK on the frozen image, followed by the one-line takeaway about freezing filesystem + runtime + deps.

## 2 - Image = layers, and the build cache

An image is a stack of layers, one per Dockerfile instruction, each keyed by a content hash. A cache miss on one layer invalidates every layer below it - so you order stable things first and volatile things last. This cell rebuilds the same Dockerfile three times and shows the cache cascade, then contrasts the deps-before-code order against the anti-pattern.

In [ ]:
import hashlib
def h(*parts):
    return hashlib.md5("|".join(parts).encode()).hexdigest()[:8]

# A Dockerfile is an ORDERED list of instructions; each makes a LAYER keyed by a hash
# of (the layer below + the instruction + its inputs). A cache MISS cascades downward.
def build(instructions, prev):
    layers, cache, parent, broke = [], {}, "root", False
    for name, content in instructions:
        key = h(parent, name, content)
        if not broke and prev.get(name) == key:
            status = "CACHED"
        else:
            status, broke = "BUILD ", True   # a miss invalidates every layer below
        layers.append((status, key, name)); cache[name] = key; parent = key
    return layers, cache

def show(title, layers):
    built = sum(1 for s, _, _ in layers if s.strip() == "BUILD")
    print(title)
    for status, key, name in layers:
        print("  [" + status + "] " + key + "  " + name)
    print("  -> rebuilt " + str(built) + " layer(s)\n")

def right(code, reqs):   # deps BEFORE code (correct)
    return [("FROM  python:3.13-slim", "base"),
            ("COPY  requirements.txt", reqs),
            ("RUN   pip install -r requirements.txt", reqs),
            ("COPY  . .   (app code)", code),
            ("CMD   [python, main.py]", "cmd")]

l1, c1 = build(right("v1", "r1"), {});  show("Build 1 (cold cache): everything builds", l1)
l2, c2 = build(right("v2", "r1"), c1);  show("Build 2 (edited code, deps unchanged): deps stay cached", l2)
l3, c3 = build(right("v2", "r2"), c2);  show("Build 3 (edited requirements.txt): reinstall cascades", l3)

def wrong(code, reqs):   # code BEFORE deps (anti-pattern)
    return [("FROM  python:3.13-slim", "base"),
            ("COPY  . .   (app code)", code),
            ("COPY  requirements.txt", reqs),
            ("RUN   pip install -r requirements.txt", reqs),
            ("CMD   [python, main.py]", "cmd")]

w1, wc1 = build(wrong("v1", "r1"), {})
w2, wc2 = build(wrong("v2", "r1"), wc1)  # ONLY the code changed
show("Anti-pattern (code before deps), edit code: pip reinstalls AGAIN", w2)

# Output:
# Build 1 (cold cache): everything builds
#   [BUILD ] 2bc9207b  FROM  python:3.13-slim
#   [BUILD ] 215238fa  COPY  requirements.txt
#   [BUILD ] 97beb503  RUN   pip install -r requirements.txt
#   [BUILD ] d245c64e  COPY  . .   (app code)
#   [BUILD ] 8248bd5e  CMD   [python, main.py]
#   -> rebuilt 5 layer(s)
#
# Build 2 (edited code, deps unchanged): deps stay cached
#   [CACHED] 2bc9207b  FROM  python:3.13-slim
#   [CACHED] 215238fa  COPY  requirements.txt
#   [CACHED] 97beb503  RUN   pip install -r requirements.txt
#   [BUILD ] 32c89dc0  COPY  . .   (app code)
#   [BUILD ] 88650a5a  CMD   [python, main.py]
#   -> rebuilt 2 layer(s)
#
# Build 3 (edited requirements.txt): reinstall cascades
#   [CACHED] 2bc9207b  FROM  python:3.13-slim
#   [BUILD ] 24c29eb3  COPY  requirements.txt
#   [BUILD ] 377f7251  RUN   pip install -r requirements.txt
#   [BUILD ] fd5dde5d  COPY  . .   (app code)
#   [BUILD ] feb03504  CMD   [python, main.py]
#   -> rebuilt 4 layer(s)
#
# Anti-pattern (code before deps), edit code: pip reinstalls AGAIN
#   [CACHED] 2bc9207b  FROM  python:3.13-slim
#   [BUILD ] 4e981a45  COPY  . .   (app code)
#   [BUILD ] 89604e23  COPY  requirements.txt
#   [BUILD ] 41ff6a67  RUN   pip install -r requirements.txt
#   [BUILD ] 2afcfa54  CMD   [python, main.py]
#   -> rebuilt 4 layer(s)

**Explanation**

A simulator of Docker's layer cache, not a real build. `build` walks an ordered instruction list, hashing each layer from (parent hash + instruction + inputs), and the moment one layer misses it flips `broke` so every layer below also rebuilds. Read it as: the cache is a chain, and a break cascades downward.

**How the code works - step by step**
- **`h`** - an 8-char md5 helper that stands in for a layer's content hash.
- **`build`** - iterates instructions, marks a layer CACHED only if its previous hash matches and nothing above it broke, otherwise BUILD (and sets `broke=True` so the miss cascades).
- **`show`** - prints each layer's status/hash/name and counts how many rebuilt.
- **`right`** - the correct Dockerfile: deps (requirements + install) *before* the app-code COPY.
- **Build 1/2/3** - cold cache (all 5 build), then a code edit (only code+CMD rebuild), then a requirements edit (the miss cascades through install and below).
- **`wrong`** - the anti-pattern with `COPY . .` before deps; a one-line code edit forces pip to reinstall everything.

**In one line:** order stable-before-volatile so a code edit rebuilds only the code layer, not the whole install.

**What you'll see:** Four labeled builds print their five layers each as [CACHED] or [BUILD] with hashes: Build 1 rebuilds 5, Build 2 rebuilds 2 (deps stay cached), Build 3 rebuilds 4 (requirements-edit cascade), and the anti-pattern rebuilds 4 on a code-only edit - proving the wrong order reinstalls every package.

## 3 - Base image and build context

Every image starts `FROM` a base, and that one line sets your floor for size, compatibility, and CVEs. This cell compares full/slim/alpine/distroless on final image size, CVE count, and cold-start pull time, then shows how a `.dockerignore` shrinks the build context you send to the daemon.

In [ ]:
# Base-image sizes (MB, uncompressed, illustrative 2026 figures).
BASE = {
    "python:3.13 (full)":   {"size": 1020, "libc": "glibc", "cve": 120, "note": "everything, huge"},
    "python:3.13-slim":     {"size": 150,  "libc": "glibc", "cve": 25,  "note": "glibc, AI wheels just work"},
    "python:3.13-alpine":   {"size": 50,   "libc": "musl",  "cve": 8,   "note": "musl -> compiles numpy from source"},
    "distroless/python3":   {"size": 50,   "libc": "glibc", "cve": 2,   "note": "no shell, minimal attack surface"},
}
VENV, CODE, SPEED = 180, 5, 50   # deps MB, code MB, pull MB/s

print("Base image -> final image size, CVEs, and cold-start pull time:")
print("  {:<22}{:>7}{:>6}{:>9}   {}".format("base", "img MB", "CVEs", "pull", "note"))
for name, b in BASE.items():
    total = b["size"] + VENV + CODE
    print("  {:<22}{:>7}{:>6}{:>9}   {}".format(name, total, b["cve"], format(total / SPEED, ".1f") + "s", b["note"]))
print()
print("For AI: slim is the sweet spot - glibc means numpy/torch install as prebuilt wheels;")
print("alpine's musl forces a from-source compile; distroless/Chainguard cut CVEs further.")
print()

# .dockerignore: what gets sent to the daemon as the build CONTEXT.
context = {"app code": 5, ".git/": 120, ".venv/": 180, "__pycache__/": 15, "tests/": 20, ".env (secret!)": 1}
full_ctx, kept_ctx = sum(context.values()), 5
print(".dockerignore - the build context sent to the daemon:")
print("  without .dockerignore: {} MB  ({})".format(full_ctx, ", ".join(context)))
print("  with    .dockerignore: {} MB  (app code only; .git/.venv/__pycache__/.env excluded)".format(kept_ctx))
print("  -> {}x smaller context, faster uploads, and no .env leaking into a layer".format(round(full_ctx / kept_ctx)))

# Output:
# Base image -> final image size, CVEs, and cold-start pull time:
#   base                   img MB  CVEs     pull   note
#   python:3.13 (full)       1205   120    24.1s   everything, huge
#   python:3.13-slim          335    25     6.7s   glibc, AI wheels just work
#   python:3.13-alpine        235     8     4.7s   musl -> compiles numpy from source
#   distroless/python3        235     2     4.7s   no shell, minimal attack surface
#
# For AI: slim is the sweet spot - glibc means numpy/torch install as prebuilt wheels;
# alpine's musl forces a from-source compile; distroless/Chainguard cut CVEs further.
#
# .dockerignore - the build context sent to the daemon:
#   without .dockerignore: 341 MB  (app code, .git/, .venv/, __pycache__/, tests/, .env (secret!))
#   with    .dockerignore: 5 MB  (app code only; .git/.venv/__pycache__/.env excluded)
#   -> 68x smaller context, faster uploads, and no .env leaking into a layer

**Explanation**

A comparison harness over a small lookup table of illustrative 2026 base-image figures - no real image is pulled. It computes each base's final image size (base + venv + code), its CVE count, and a pull time, then models the build-context shrink from a `.dockerignore`.

**How the code works - step by step**
- **`BASE`** - a dict of four bases with size, libc (glibc vs musl), CVE count, and a one-line note.
- **`VENV, CODE, SPEED`** - fixed deps/code sizes and a pull-rate (MB/s) used for every row.
- **The print loop** - adds venv+code to each base and prints size, CVEs, and `total/SPEED` as a cold-start pull time; the notes flag that alpine's musl compiles numpy from source.
- **`context`** - a dict of what sits in your directory (app code, `.git`, `.venv`, `__pycache__`, `tests`, a stray `.env`).
- **`full_ctx` vs `kept_ctx`** - the sum of everything versus app-code-only, and the ratio printed as the context-shrink factor.

**In one line:** slim is the AI default (glibc = prebuilt wheels), and a `.dockerignore` keeps `.git`/`.venv`/`.env` out of the context.

**What you'll see:** A table maps each base to final image MB, CVEs, and pull seconds (full 1205MB/24.1s down to slim 335MB/6.7s and distroless 235MB with just 2 CVEs), then the `.dockerignore` block shows the context dropping from 341 MB to 5 MB - about 68x smaller with no `.env` leaking into a layer.

## 4 - Multi-stage: build heavy, ship light

Installing AI deps often needs a compiler, but you don't need that compiler to *run* the app. A multi-stage build compiles in a throwaway `builder` stage and copies only the finished venv into a clean `runtime` stage. This cell computes both image sizes side by side.

In [ ]:
BASE, BUILD_TOOLS, VENV, APT_CACHE, CODE = 150, 200, 180, 45, 5   # MB (illustrative)

single = BASE + BUILD_TOOLS + VENV + APT_CACHE + CODE
print("SINGLE-STAGE (build tools ship to production):")
print("  base {} + build-tools {} + venv {} + apt-cache {} + code {} = {} MB".format(
    BASE, BUILD_TOOLS, VENV, APT_CACHE, CODE, single))
print("  build tools in the final image: gcc, make (2)")
print()

# Multi-stage: the builder compiles; the runtime copies ONLY the venv.
builder = BASE + BUILD_TOOLS + VENV     # exists only during the build, then discarded
runtime = BASE + VENV + CODE
print("MULTI-STAGE (builder compiles, runtime ships only the venv):")
print("  builder (discarded): base {} + build-tools {} + venv {} = {} MB".format(BASE, BUILD_TOOLS, VENV, builder))
print("  runtime  (shipped):  base {} + venv {} + code {} = {} MB".format(BASE, VENV, CODE, runtime))
print("  build tools in the final image: none (0)")
print()
print("  saved {} MB ({}% smaller), and 0 compilers in production = smaller attack surface".format(
    single - runtime, round((single - runtime) / single * 100)))

# Output:
# SINGLE-STAGE (build tools ship to production):
#   base 150 + build-tools 200 + venv 180 + apt-cache 45 + code 5 = 580 MB
#   build tools in the final image: gcc, make (2)
#
# MULTI-STAGE (builder compiles, runtime ships only the venv):
#   builder (discarded): base 150 + build-tools 200 + venv 180 = 530 MB
#   runtime  (shipped):  base 150 + venv 180 + code 5 = 335 MB
#   build tools in the final image: none (0)
#
#   saved 245 MB (42% smaller), and 0 compilers in production = smaller attack surface

**Explanation**

An arithmetic model of image composition - it just adds up MB for each stage's contents. The single-stage total drags gcc/make and the apt cache into production; the multi-stage runtime keeps only base + venv + code, and the builder (with the compilers) is discarded.

**How the code works - step by step**
- **The MB constants** - base, build tools, venv, apt cache, and code, all illustrative.
- **`single`** - sums every piece, so build tools and the apt cache ship to production (gcc + make counted in the final image).
- **`builder`** - base + build tools + venv; exists only during the build, then thrown away.
- **`runtime`** - base + venv + code on a fresh slim base, with zero build tools.
- **The savings line** - prints the MB saved and percent smaller, plus the point that 0 compilers in prod means a smaller attack surface.

**In one line:** compile in a discarded builder, copy only the venv into runtime - smaller image, no compilers shipped.

**What you'll see:** The single-stage image totals 580 MB with gcc/make inside; the multi-stage runtime ships 335 MB with none, printing "saved 245 MB (42% smaller)" and zero build tools in production.

## 5 - Reproducible and fast dependencies with uv

Two dependency problems, one step. Unpinned requirements drift to a different version next week (the exact cause of "works on my machine"); and pip is slow when CI builds many times a day. This cell shows the drift, then races pip against uv and a BuildKit cache mount on install time.

In [ ]:
# What "latest" resolves to on two different days (illustrative index).
INDEX = {
    "day-1": {"fastapi": "0.115.6", "uvicorn": "0.34.0", "pydantic": "2.10.5"},
    "day-2": {"fastapi": "0.118.0", "uvicorn": "0.37.1", "pydantic": "2.11.9"},
}
def resolve(spec, day):
    name, _, pin = spec.partition("==")
    return pin if pin else INDEX[day][name]

print("Reproducibility - resolve the same requirement on two days:")
for spec in ["fastapi", "fastapi==0.115.6"]:
    d1, d2 = resolve(spec, "day-1"), resolve(spec, "day-2")
    tag = "SAME (reproducible)" if d1 == d2 else "DRIFTED (works on my machine!)"
    print("  {:<20} day-1={:<9} day-2={:<9} -> {}".format(spec, d1, d2, tag))
print()

# Install speed: 8 packages, cold pip vs uv vs a BuildKit cache mount.
N, pip_each, uv_each, cache_each = 8, 4.0, 0.40, 0.05
print("Install speed - {} packages:".format(N))
print("  pip (cold)                 {:>6.1f}s".format(N * pip_each))
print("  uv  (cold, Rust resolver)  {:>6.1f}s   ({}x faster than pip)".format(N * uv_each, round(pip_each / uv_each)))
print("  uv  + BuildKit cache mount {:>6.1f}s   (wheels already downloaded)".format(N * cache_each))
print()
print("Pin every version and commit a lock file for reproducibility; use uv + a cache mount for speed.")

# Output:
# Reproducibility - resolve the same requirement on two days:
#   fastapi              day-1=0.115.6   day-2=0.118.0   -> DRIFTED (works on my machine!)
#   fastapi==0.115.6     day-1=0.115.6   day-2=0.115.6   -> SAME (reproducible)
#
# Install speed - 8 packages:
#   pip (cold)                   32.0s
#   uv  (cold, Rust resolver)     3.2s   (10x faster than pip)
#   uv  + BuildKit cache mount    0.4s   (wheels already downloaded)
#
# Pin every version and commit a lock file for reproducibility; use uv + a cache mount for speed.

**Explanation**

Two small simulations sharing a cell. The first resolves a requirement against a two-day fake package index to expose version drift; the second multiplies a per-package install time to compare pip, uv, and a warm cache mount. Both are illustrative models, not real installs.

**How the code works - step by step**
- **`INDEX`** - what "latest" resolves to on day-1 vs day-2 for a few packages.
- **`resolve`** - returns the pinned version if the spec has `==`, otherwise looks up that day's latest.
- **The reproducibility loop** - resolves `fastapi` (unpinned) and `fastapi==0.115.6` on both days and tags each SAME or DRIFTED.
- **`N, pip_each, uv_each, cache_each`** - 8 packages and a per-package second-cost for each installer.
- **The speed block** - prints pip cold, uv cold (with the x-faster multiple), and uv + BuildKit cache mount (wheels already downloaded).

**In one line:** pin every version + commit a lock file for reproducibility; use uv + a cache mount for speed.

**What you'll see:** Unpinned `fastapi` shows DRIFTED (0.115.6 -> 0.118.0) while the pinned spec stays SAME on both days; the install race prints pip 32.0s, uv 3.2s (10x faster), and uv + cache mount 0.4s.

## 6 - Run it safely: non-root, PID 1, healthcheck

A small reproducible image can still be a production hazard if it runs wrong. This cell models three safety details: the blast radius of root vs non-root, why the exec-form CMD (your app as PID 1) drains requests on SIGTERM while the shell form drops them, and how a healthcheck restarts an unhealthy instance after three strikes.

In [ ]:
# 1) Blast radius: root vs non-root if the container is breached.
def blast(uid):
    return "FULL host access (write anywhere, install, escalate)" if uid == 0 else "limited to the app's own files"
print("Non-root - the blast radius if the container is breached:")
print("  USER root    (uid 0)    -> " + blast(0))
print("  USER appuser (uid 1000) -> " + blast(1000))
print()

# 2) Graceful shutdown. Docker sends SIGTERM to pid 1, waits 10s, then SIGKILL.
def deploy(cmd_form):
    inflight = 3
    if cmd_form == "exec":   # CMD ["python","main.py"] -> python IS pid 1, handles SIGTERM
        pid1, handled = "python (your app)", True
    else:                    # CMD python main.py -> /bin/sh is pid 1, does not forward SIGTERM
        pid1, handled = "/bin/sh", False
    outcome = "drained gracefully" if handled else "KILLED at the 10s SIGKILL - requests dropped"
    print("  CMD {:<6} pid1={:<18} SIGTERM handled={!s:<5} -> {} in-flight {}".format(
        cmd_form, pid1, handled, inflight, outcome))

print("Graceful shutdown - a rolling deploy sends SIGTERM to pid 1:")
deploy("shell")
deploy("exec")
print()

# 3) Healthcheck: three consecutive failures restart the instance.
print("Healthcheck - three strikes restart the instance:")
fails = 0
for i, code in enumerate([200, 200, 500, 500, 500], 1):
    fails = 0 if code == 200 else fails + 1
    action = "RESTART (3 strikes)" if fails >= 3 else "ok"
    print("  probe {} -> HTTP {}  fails={}  {}".format(i, code, fails, action))

# Output:
# Non-root - the blast radius if the container is breached:
#   USER root    (uid 0)    -> FULL host access (write anywhere, install, escalate)
#   USER appuser (uid 1000) -> limited to the app's own files
#
# Graceful shutdown - a rolling deploy sends SIGTERM to pid 1:
#   CMD shell  pid1=/bin/sh            SIGTERM handled=False -> 3 in-flight KILLED at the 10s SIGKILL - requests dropped
#   CMD exec   pid1=python (your app)  SIGTERM handled=True  -> 3 in-flight drained gracefully
#
# Healthcheck - three strikes restart the instance:
#   probe 1 -> HTTP 200  fails=0  ok
#   probe 2 -> HTTP 200  fails=0  ok
#   probe 3 -> HTTP 500  fails=1  ok
#   probe 4 -> HTTP 500  fails=2  ok
#   probe 5 -> HTTP 500  fails=3  RESTART (3 strikes)

**Explanation**

Three independent simulations of runtime behavior, none of them starting a real process. `blast` maps a uid to a damage description, `deploy` models who is PID 1 and whether SIGTERM is handled for each CMD form, and a small loop counts consecutive healthcheck failures until a restart.

**How the code works - step by step**
- **`blast`** - returns full-host-access for uid 0, else limited-to-app-files, printed for root vs appuser.
- **`deploy`** - for the exec form your app (python) is PID 1 and handles SIGTERM (drains); for the shell form `/bin/sh` is PID 1, doesn't forward SIGTERM, and the 3 in-flight requests are killed at the 10s SIGKILL.
- **The healthcheck loop** - walks probe results `[200,200,500,500,500]`, resetting the fail counter on 200 and restarting once three consecutive failures accumulate.

**In one line:** non-root shrinks the blast radius, exec-form CMD drains on SIGTERM, and three failed probes trigger a restart.

**What you'll see:** Prints that root gives full host access vs appuser's limited files; the shell-form deploy KILLS 3 in-flight requests at the 10s SIGKILL while the exec form drains them gracefully; and the probe loop reaches fails=3 on probe 5 and prints RESTART (3 strikes).

## 7 - Ship it: build, scan, push, run

Now put the image on the internet. The pipeline is four steps - build (BuildKit), scan for CVEs (Trivy/Docker Scout), push only the changed layers to a registry, run it on a platform like Cloud Run. This cell runs a fat single-stage pipeline against a slim multi-stage one to show how every earlier choice compounds.

In [ ]:
# Two pipelines, end to end: build -> scan -> push -> run.
def pipeline(name, img_mb, cve, push_mb, push_note):
    cold = img_mb / 100.0    # cold start roughly scales with image size (illustrative)
    print("  " + name)
    print("    build -> image {} MB".format(img_mb))
    print("    scan  -> {} known CVEs".format(cve))
    print("    push  -> {} MB uploaded ({})".format(push_mb, push_note))
    print("    run   -> ~{:.1f}s cold start".format(cold))
    return {"mb": img_mb, "cve": cve, "push": push_mb, "cold": cold}

print("Pipeline A - fat, single-stage, unscanned base:")
a = pipeline("A", 1205, 120, 1205, "first push - every layer")     # full base, all layers
print()
print("Pipeline B - slim, multi-stage, scanned:")
b = pipeline("B", 335, 25, 5, "only the changed code layer")       # deps layer already in the registry
print()
print("Slim multi-stage vs fat single-stage:")
print("  image   {} MB -> {} MB   ({}% smaller)".format(a["mb"], b["mb"], round((a["mb"] - b["mb"]) / a["mb"] * 100)))
print("  CVEs    {} -> {}".format(a["cve"], b["cve"]))
print("  push    {} MB -> {} MB   (layer dedup)".format(a["push"], b["push"]))
print("  cold    {:.1f}s -> {:.1f}s".format(a["cold"], b["cold"]))
print("  Every earlier step compounds: base image -> size -> CVEs -> push time -> cold start.")

# Output:
# Pipeline A - fat, single-stage, unscanned base:
#   A
#     build -> image 1205 MB
#     scan  -> 120 known CVEs
#     push  -> 1205 MB uploaded (first push - every layer)
#     run   -> ~12.1s cold start
#
# Pipeline B - slim, multi-stage, scanned:
#   B
#     build -> image 335 MB
#     scan  -> 25 known CVEs
#     push  -> 5 MB uploaded (only the changed code layer)
#     run   -> ~3.4s cold start
#
# Slim multi-stage vs fat single-stage:
#   image   1205 MB -> 335 MB   (72% smaller)
#   CVEs    120 -> 25
#   push    1205 MB -> 5 MB   (layer dedup)
#   cold    12.1s -> 3.4s
#   Every earlier step compounds: base image -> size -> CVEs -> push time -> cold start.

**Explanation**

A reporting harness that prints the four pipeline stages for two images and then diffs them. `pipeline` derives a cold-start time from image size and echoes the scan/push figures you feed it; the comparison at the end is where the compounding shows up.

**How the code works - step by step**
- **`pipeline`** - prints build (image MB), scan (CVE count), push (MB uploaded + note), and run (cold start ~ image_mb/100), returning the numbers for later diffing.
- **Pipeline A** - fat single-stage: 1205 MB, 120 CVEs, first push of every layer, slow cold start.
- **Pipeline B** - slim multi-stage: 335 MB, 25 CVEs, pushes only the changed code layer (deps already in the registry), fast cold start.
- **The comparison block** - prints the percent-smaller image, the CVE drop, the push-MB drop from layer dedup, and the cold-start drop.

**In one line:** build -> scan -> push -> run, where a smaller layer-ordered image scans cleaner, pushes in megabytes, and cold-starts in seconds.

**What you'll see:** Two four-line pipelines print in full (A at 1205 MB/120 CVEs/12.1s, B at 335 MB/25 CVEs/3.4s), then the diff: image 72% smaller, CVEs 120->25, push 1205 MB->5 MB via layer dedup, cold 12.1s->3.4s.

## 8 - The production Dockerfile (reference)

Everything from Steps 1-7 assembled into one illustrative Dockerfile plus the shell commands to ship it. It's written to disk with `%%writefile` so you can read it as a file - it is not built or run here (that needs Docker, a `uv.lock`, and gcloud).

In [ ]:
%%writefile Dockerfile
# THE PRODUCTION DOCKERFILE - an illustrative minimal example (multi-stage + uv + non-root + healthcheck).
# syntax=docker/dockerfile:1

# ---- STAGE 1: builder (compiles deps with uv, then is discarded) ----
FROM python:3.13-slim AS builder
COPY --from=ghcr.io/astral-sh/uv:0.8.4 /uv /uvx /usr/local/bin/   # pin uv too - Step 5: pin every input
ENV UV_COMPILE_BYTECODE=1 UV_LINK_MODE=copy      # precompile .pyc; copy (not hardlink) across the cache mount
WORKDIR /app
# deps FIRST (a cached layer), with a BuildKit cache mount for uv's download cache
COPY pyproject.toml uv.lock ./
RUN --mount=type=cache,target=/root/.cache/uv uv sync --frozen --no-install-project --no-editable
COPY . .
RUN --mount=type=cache,target=/root/.cache/uv uv sync --frozen --no-editable

# ---- STAGE 2: runtime (ships only the venv + code, no build tools) ----
FROM python:3.13-slim AS runtime
RUN useradd --create-home appuser
COPY --from=builder --chown=appuser /app/.venv /app/.venv
COPY --from=builder --chown=appuser /app /app
ENV PATH="/app/.venv/bin:$PATH" PYTHONUNBUFFERED=1
WORKDIR /app
USER appuser                                     # non-root: shrink the blast radius
EXPOSE 8080
HEALTHCHECK --interval=30s --timeout=5s --retries=3 \
  CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8080/health')" || exit 1
# exec form: uvicorn is pid 1 and receives SIGTERM -> drains in-flight requests
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"]

# --- Build, scan, and ship (shell) ---
#   docker build -t netsetos-ai:$(git rev-parse --short HEAD) .    # tag with the git SHA (immutable), never :latest
#   docker run --rm -p 8080:8080 netsetos-ai:...     # smoke-test locally first, then: curl localhost:8080/health
#   trivy image netsetos-ai:...            # or: docker scout cves netsetos-ai:...
#   gcloud run deploy netsetos-ai --source . --region asia-south1 --port 8080 --set-secrets ANTHROPIC_API_KEY=...
# Output: (illustrative - needs Docker + a uv.lock + gcloud; sizes and CVE counts vary by machine.)

**Explanation**

A reference artifact, not an executed cell: `%%writefile Dockerfile` dumps the text to a file. Read it as the whole lesson in Dockerfile form - a builder stage that compiles with uv, a runtime stage that ships only the venv, non-root, healthcheck, exec-form CMD, and the build/scan/deploy commands as comments.

**How the code works - step by step**
- **Stage 1 `builder`** - starts from `python:3.13-slim`, copies a pinned `uv` binary, sets `UV_COMPILE_BYTECODE=1`, copies `pyproject.toml`/`uv.lock` first, and `uv sync --frozen` behind a BuildKit cache mount (Steps 2, 4, 5).
- **Stage 2 `runtime`** - fresh slim base, creates `appuser`, copies only the `.venv` and code `--from=builder`, and switches to `USER appuser` (Steps 3, 4, 6).
- **`HEALTHCHECK`** - probes `/health` every 30s and fails after 3 retries (Step 6).
- **Exec-form `CMD`** - `["uvicorn", ...]` so uvicorn is PID 1 and drains on SIGTERM.
- **The comment block** - the ship step: `docker build` tagged with the git SHA, a local smoke test, `trivy`/`docker scout` scan, then `gcloud run deploy --source` with runtime `--set-secrets`.

**In one line:** the Dockerfile reads top-to-bottom as Steps 1-7 - multi-stage + uv + non-root + healthcheck, then build/scan/push/run.

**What you'll see:** Jupyter prints "Writing Dockerfile" (or "Overwriting Dockerfile"); no program runs. The final comment marks the block as illustrative - actually building it needs Docker, a uv.lock, and gcloud, and sizes/CVE counts vary by machine.

Across seven small keyless simulations you built the full container discipline: freeze the environment into an image so it runs identically everywhere, order layers stable-before-volatile to keep the build cache warm, pick a slim (glibc) base and a clean build context, compile in a throwaway builder and ship only the venv, pin+lock deps and install them fast with uv, run non-root with an exec-form CMD and a healthcheck, then build/scan/push/run where every earlier choice compounds. Ask four questions of any image - is it reproducible, small, safe, and healthy. Next up is Lesson 12.6 (CI/CD that runs this pipeline on every push) and 12.7 (scaling many containers with Kubernetes).